# Chapter 13: Information and privacy audits

Run all cells in order. Python 3.12.13 and NumPy 2.0.2 were used for the recorded outputs. These are the five complete manuscript listings; no quantum service or private data is used.

The fixed two-qubit classifier applies RY(x1), RY(x2), then CNOT(0,1), with basis order 00, 01, 10, 11. The score is (1 - <Z1>)/2. The 16-point synthetic grid is a diagnostic, not a generalization benchmark or evidence of quantum advantage. The shot experiment uses independent Born samples with seed 13. The privacy cell calculates an ideal mechanism's sensitivity, noise scales, and accounting; it is not a production privacy mechanism.


## 1. Aliasing, overlap, and purity

The probe a = (pi/3, pi/3) and its 2pi alias have the same density matrix. Compare a with b = (-pi/3, pi/3), then apply global depolarization with weight 0.3. Mixed self-overlap is purity, while self-fidelity is one.

In [ ]:
import numpy as np


def state(x, entangle=True):
    """Return the two-angle state, optionally followed by CNOT(0, 1)."""
    q0 = np.array([np.cos(x[0]/2), np.sin(x[0]/2)])
    q1 = np.array([np.cos(x[1]/2), np.sin(x[1]/2)])
    psi = np.kron(q0, q1)
    return psi[[0, 1, 3, 2]] if entangle else psi


def density(psi):
    """Return the density matrix of a normalized statevector psi."""
    return np.outer(psi, psi.conj())


def trace_distance(rho, sigma):
    """Return trace distance for two Hermitian density matrices."""
    return np.abs(np.linalg.eigvalsh(rho - sigma)).sum() / 2


angles = np.pi * np.array([1/6, 1/3, 2/3, 5/6])
grid = np.array([(u, v) for u in angles for v in angles])
labels = (np.cos(grid).prod(axis=1) < 0).astype(int)
a = np.array([np.pi/3, np.pi/3])
b = np.array([-np.pi/3, np.pi/3])
rho_a, rho_b = density(state(a)), density(state(b))
rho_alias = density(state(a + [2*np.pi, 0]))
mixed_a = 0.7*rho_a + 0.3*np.eye(4)/4
mixed_b = 0.7*rho_b + 0.3*np.eye(4)/4
print("Alias distance:", round(trace_distance(rho_a, rho_alias), 6))
print("Pure overlap:", round(np.trace(rho_a @ rho_b).real, 6))
print("Trace distance:", round(trace_distance(rho_a, rho_b), 6))
print("Mixed purity:", round(np.trace(mixed_a @ mixed_a).real, 6))
print("Mixed distance:", round(trace_distance(mixed_a, mixed_b), 6))


## 2. Readout audit

Computational-basis readout cannot distinguish the probe pair. XX can attain the equal-prior Helstrom success probability for this pair. Both probes have the same classifier label, so this phase distinction is not needed for the target.

In [ ]:
I2 = np.eye(2)
Z = np.diag([1., -1.])
X = np.array([[0., 1.], [1., 0.]])
Z0, Z1, XX = np.kron(Z, I2), np.kron(I2, Z), np.kron(X, X)


def expectation(x, observable=Z1, entangle=True):
    """Return the exact observable mean for angles x and the chosen circuit."""
    psi = state(x, entangle)
    return float(np.vdot(psi, observable @ psi).real)


basis_tv = np.abs(np.diag(rho_a - rho_b)).sum()/2
xx_tv = abs(expectation(a, XX) - expectation(b, XX))/2
mu = np.array([expectation(x) for x in grid])
scores = (1 - mu)/2
print("Z-basis TV / XX TV:", round(basis_tv, 6), round(xx_tv, 6))
print("Exact grid accuracy:", np.mean((scores >= 0.5) == labels))


## 3. Shot uncertainty

Use 1,000 independent batches for each budget. A score of exactly 0.5 predicts class 1. Columns: shots per input, theoretical score SE at a, mean grid accuracy, and sample SD across batches. This is conditional measurement uncertainty, not population uncertainty. Binomial sampling aggregates counts representing 37,376,000 preparations.

In [ ]:
rng = np.random.default_rng(13)
shot_rows = []
for shots in (32, 256, 2048):
    estimates = rng.binomial(shots, scores, size=(1000, 16))/shots
    accuracies = ((estimates >= 0.5) == labels).mean(axis=1)
    se_a = np.sqrt(0.375 * 0.625 / shots)
    row = (shots, se_a, accuracies.mean(), accuracies.std(ddof=1))
    shot_rows.append(row)
    print(f"{shots:4d}  {se_a:.6f}  {row[2]:.6f}  {row[3]:.6f}")


## 4. Sensitivity and ablations

Finite differences use exact expectations. The classical cosine product reproduces every quantum score. Removing the CNOT or measuring only Z0 loses the prescribed interaction.

In [ ]:
step = 1e-5
shifts = step*np.eye(2)
numeric = np.array([
    (expectation(a + dx) - expectation(a - dx))/(2*step)
    for dx in shifts
])
print("Score sensitivity:", np.round(np.abs(numeric)/2, 6))
ablation_means = {
    "Full Z1": mu,
    "Read Z0": np.array([expectation(x, Z0) for x in grid]),
    "No CNOT": np.array([expectation(x, Z1, False) for x in grid]),
    "Classical cosine product": np.cos(grid).prod(axis=1),
}
ablation_accuracies = {}
for name, values in ablation_means.items():
    accuracy = float(np.mean(((1 - values)/2 >= 0.5) == labels))
    ablation_accuracies[name] = accuracy
    print(name, accuracy)


## 5. Privacy accounting and utility

For a public fixed classifier and N=100 private records, replace-one adjacency gives sensitivity 1/N for either mean score or accuracy. Each person contributes one bounded record. Two ideal Laplace releases with epsilon=0.5 each compose to (1,0)-DP. The printed error allowances are per-release Laplace noise bounds, not population confidence intervals. The code only calculates scales and budgets; it does not release records or draw privacy noise. Production implementations require protected randomness, finite-precision safeguards, and budget enforcement. The seed above is not suitable for privacy noise.

In [ ]:
cohort_size = 100
sensitivity = 1/cohort_size
release_budget = np.array([0.5, 0.5])
privacy_rows = []
for epsilon in (1.0, 0.5, 0.1):
    scale = sensitivity/epsilon
    absolute_error_95 = scale*np.log(20)
    privacy_rows.append((epsilon, scale, absolute_error_95))
    print(f"{epsilon:.1f}  {scale:.3f}  {absolute_error_95:.6f}")
print("Two-release epsilon:", release_budget.sum())
print("Two-release delta:", 0)
